# Introduction to data analysis in Cosmology: Fourier Transforms and Spherical Harmonics


In this notebook, we are going to introduce Fourier transorms, and in particular Discrete Fourier Trasnforms, that will be useful for data analysis. 

We will introduce 1D and 2D transforms, as well as image filtering.


# 0. Preliminaries

Some useful shortcuts with jupyter notebook: 
- SHIFT+ ENTER in a cell to run the cell 
- SHIFT+TAB when the cursor is inside a function to show its documentation 

# 1. Fourier Transforms

The Fourier Transform is a decomposition of a signal (for instance a time series, a sound, or an image), into basic building blocks, which are sinusoids of different frenquency. This allows to decompose the signal from the time (or spatial) domain, into the frequency domain.  


The image below illustrate this: on the left the signal, in time domain, and on the right its Fourier transform, in frequency domain. The signal can be decomposed as a sum of three sinusoids, each with a specific frequency, as can be seen on the frequency domain plot. 

<img src="https://upload.wikimedia.org/wikipedia/commons/6/61/FFT-Time-Frequency-View.png">

In the 1D and continuous case, the Fourier transform is expressed as

\begin{equation}
\tilde{f}(k) = \int_{- \infty }^{+\infty} f(x) e^{-2\pi ikx} dx
\end{equation} 

and the inverse transform, who gives back the original signal, is given by:

\begin{equation}
f(x) = \int_{-\infty}^{+\infty} \tilde{f}(k) e^{2\pi ikx} dk
\end{equation} 

where $f(x)$ is the signal in real space, $\tilde{f}(k)$ is the signal in Fourier space, k is the frequency, and the exponential describes the sinusoids composing the signal, in complex notation: $e^{ix} = \cos(x) + i \sin(x)$


One useful property of the FT is:
$$f(x) \in  \mathbb{R} \Leftrightarrow \tilde f^*(k) = \tilde f(-k)$$ where $*$ is the complex conjuguation: $z = x+iy \Rightarrow z^* = x - i y$, with $\{x, y\} \in \mathbb{R}$. In plain words, the FT of a real function in time domain is a conjuguate symetric function in Fourier space, and conversely, a conjugate-symmetric Fourier-space function has a real inverse Fourier transform.

This means that if $f(x)$ is real and even, i.e. $f(x)=f(-x)$, then $\tilde f(k)$ is also real and even. 

Another useful property is that the FT of a product of functions in time domain is a convolution in Fourier space.


## First example: A Gaussian

The classic example is a Gaussian, whose Fourier transform is a Gaussian as well:

\begin{equation}
g(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \, e^{- x^2 / (2\sigma^2)}
\qquad\Longrightarrow\qquad
\tilde{g}(k) =  e^{-2 \pi^2  \sigma^2 k^2}
\end{equation}

It is one of the few cases where we know the answer analytically, so we can check the numerics exactly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

In [ ]:
def Fourier_transform(f, k):
    """FT integrating over the full real line. 
    Since it integrates to infinity, it is only converging numerically if f decays at large |x|.
    """
    res = np.empty(len(k), dtype=np.complex128)
    for i, _k in enumerate(k):

        integrand_real = lambda x: np.real(f(x) * np.exp(-2j * np.pi * _k * x))
        integrand_imag = lambda x: np.imag(f(x) * np.exp(-2j * np.pi * _k * x))

        real_part = quad(integrand_real, -np.inf, np.inf)[0]
        imag_part = quad(integrand_imag, -np.inf, np.inf)[0]
        res[i] = real_part + 1j * imag_part
    return res

In [ ]:
def gaussian(x, sigma=0.5):
    return 1/np.sqrt(2 * np.pi * sigma**2) * np.exp(- x**2/ (2*sigma**2))

def gaussian_ft(k, sigma=0.5):
    """Analytic Fourier transform"""
    return np.exp(-np.pi**2 * k**2 * 2 * sigma**2)


x = np.linspace(-4, 4, num=500)
k = np.linspace(-3, 3, num=121)

ft_gauss = Fourier_transform(gaussian, k)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(x, gaussian(x))
ax1.set_xlabel('x')
ax1.set_ylabel('g(x)')
ax1.set_title('real space')

ax2.plot(k, np.real(ft_gauss), lw=3, alpha=0.5, label='numerical')
ax2.plot(k, gaussian_ft(k), 'k--', label='analytic')
ax2.set_xlabel('k')
ax2.set_ylabel(r'$\tilde{g}(k)$')
ax2.set_title('Fourier space')
ax2.legend()

print("max error vs analytic:", np.abs(ft_gauss - gaussian_ft(k)).max())
print("max |imaginary part| :", np.abs(np.imag(ft_gauss)).max())

## Second example: a wave packet

The FT of a pure cosinus function is a Dirac delta function. However, it is not numerically integrable since it is periodic and does not decay at infinity.
But the combination of an oscillation at a single frequency $k_0$ multiplied by a Gaussian envelope dies away and stays integrable.

\begin{equation}
h(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-x^2 / 2\sigma^2} \cos(2\pi k_0 x)
\qquad\Longrightarrow\qquad
\tilde{h}(k) = \tfrac{1}{2}\left[ G(k - k_0) + G(k + k_0) \right],
\quad G(k) =  e^{-2\pi^2 \sigma^2 k^2}
\end{equation}

Multiplying by an envelope in real space *convolves* the transform with that envelope's transform in Fourier space. A pure $\cos(2\pi k_0 x)$ would give two Dirac deltas at $\pm k_0$; the envelope smears each one into a Gaussian of width $\propto 1/\sigma$.


#### 🧩 Compute and plot the Fourier transform of the function h(x). What do you see? 


# 2. Discrete Fourier Transforms


In real life, we do not analyse continuous signal, but very often signals are sampled. This is the case also for an image, where the pixel size define the limiting resolution of our image. 
We thus rely on a discretized version of the Fourier transform.

Let's assume we have a sequence of $N$ complex numbers $ \{{\bf{x}}_n\} := x_0, x_1, ... x_{N-1}$. Its discrete Fourier transfomr (DFT) is defined as 

$$ X_k = \sum_{n=0}^{N-1} x_n \cdot e^{-2i\pi \frac{k}{N} n} $$

each $X_{k}$ is a complex number whose polar coordinates are the amplitude and phase of a complex sinusoidal component $\left(e^{i2\pi {\frac{k}{N}}n}\right)  $ of the function $x_{n}$.

The inverse transform is given by 

$$ x_n = \frac1N \sum_{k=0}^{N-1}  X_k \cdot e^{2i\pi \frac{k}{N} n}  $$


#### Numerical considerations 


The sampling frequency is given by $f_s = 1 / \Delta x$, where $\Delta x$ is the sampling interval. 

The smallest frequency available (equivalently the frequency resolution), is $$\Delta k = \frac{f_s}{N} = \frac{1}{N \Delta x}$$
This means that the larger the physical size of our sample is, the smaller k we are able to measure. 

The largest frequency we are able to measure without aliasing is called the Nyquist frequency, and is given by $f_{\rm Nyq} = f_s /2 $. Any frequncy higher than this will not be sampled enough, and will create some aliasing: i.e. a higher frequency can be mistaken as a lower frequency. Thus, a smaller sampling step allows to you to measure higher frequencies. 


In order to compare the DFT to the continuous FT defined above, one needs to take into account the sampling interval. The DFT is then given by 
$$ X_k = \Delta x \sum_{n=0}^{N-1} x_n \cdot e^{-2i\pi \frac{k}{N} n} $$
When applying the DFT to physical data, including the sampling interval ensures correct amplitude, energy, and frequency interpretation, particularly when combining or comparing data from different sources.
This sampling interval is not present in the DFT estimated from numpy, so we have to include it explicitely when performing our FT. 



#### Fast Fourier transforms


In practice, computing naively the DFT using this algorithm is tedious: it scales as $O(n^2)$ where $n$ is the sample size, and this can be quite slow with large sample size. 
This is why we relate on fast Fourier transforms (FFT). This algorithm allows to compute the DFT with a complexity $O(n \log(n))$, which is much faster when $n$ is large.  
This algorithm is at the basis of many signal processing.
We will use the implementation from the numpy library, `np.fft`




## FFT example in 1D

Let's take back the example above, but this time as a sampled function. The sampling is defined by the range and the interval between sampled values, here called $dx$.

In [ ]:
x = np.linspace(-8, 8, num=1000)

dx = 0.3
x_samp = np.arange(-8, 8, step=dx)
size = np.size(x_samp)

wave_samp = wave_packet(x_samp) 

plt.plot(x, wave_packet(x), label='Continuous function')
plt.scatter(x_samp, wave_samp, label='Sampled values', c='red')

plt.legend()

plt.xlabel('x')
plt.ylabel('h(x)')


With numpy, for an even sample size $N$, frequencies are ordered  from 0 to $(N/2 -1)\Delta k$, and then from $(-N/2)\Delta k$ to  $-\Delta k$ (see the documentation of `np.fft.fftfreq`).

To shift the frequency components to have the $0$ in the center of the array, we can use `np.fft.fftshift`.




#### 🧩 Compute and plot the DFT of the sampled wave packet. What do you see? 

## FFT example in 2D

Now let's move to 2D space, and let's define an image by $I(x, y)$, where $x, y$ are the coordinates of the pixels, and where we have $N_x$ and $N_y$ pixels each.  It's DFT is defined by:

\begin{equation}
 \tilde{I}(k_x, k_y) = \sum_{x=0}^{N_{x}-1} \sum_{y=0}^{N_{y}-1} I(x, y) \, e^{-2\pi i (\frac{k_x  x}{N_{x}} + \frac{k_y y}{N_{y}})}
\end{equation}

where $(k_x, k_y)$ are the spatial frequencies.

In [ ]:
from skimage import data
from skimage.color import rgb2gray
from skimage import img_as_float


In [ ]:
image = img_as_float(rgb2gray(data.astronaut()))
image_ft = np.fft.fft2(image)
data_ft = np.abs(image_ft)
data_ft_shifted = np.fft.fftshift(data_ft) 



Nx, Ny = 512, 512
dx, dy = 1, 1

kx = np.fft.fftfreq(Nx, d=dx)
ky = np.fft.fftfreq(Ny, d=dy)

KX, KY = np.meshgrid(kx, ky)

R = np.sqrt(KX**2 + KY**2) 

In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
im1 = ax1.imshow(image, cmap='grey')
fig.colorbar(im1, ax=ax1)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title(r"image $f(x,y)$")


imft = ax2.imshow(np.log(np.abs(data_ft_shifted)), cmap='plasma')
fig.colorbar(imft, ax=ax2)

ax2.axes.xaxis.set_ticks([0,256,512])
ax2.axes.xaxis.set_ticklabels([r'$-\frac{1}{2\Delta x}$', '0', r'$\frac{1}{2\Delta x}$'])
ax2.set_xlabel(r'$k_x$')
ax2.axes.yaxis.set_ticks([0, 256, 512])
ax2.axes.yaxis.set_ticklabels([r'$-\frac{1}{2\Delta x}$', '0', r'$\frac{1}{2\Delta x}$'])
ax2.set_ylabel(r'$k_y$')

ax2.set_title(r"Fourier transform shifted $|F(kx,ky)|$")

### Image filtering 

FT are very useful if you want to do image processing, for instance smoothing or filtering an image. 

To filter an image in real space, you need to perfomr a convolution. This scales as $O(n^2)$, so it is quite expansive when you have millions of pixels. However, in Fourier space, a convolution is simply a multiplication, which scales as $O(n)$, so it is much faster.


Let's start to filter out the small scales from our image.

In [ ]:
def low_pass(k, kcut):
    # Keep only low frequencies
    filt = np.ones_like(k)
    filt[k > kcut] = 0
    return filt


In [ ]:
kcut = 0.05


plt.imshow(low_pass(np.fft.fftshift(R), kcut), cmap='plasma')
plt.colorbar()

In [ ]:
image_lp_ft = image_ft * low_pass(np.sqrt(KX**2 + KY**2), kcut)
image_lp = np.fft.ifft2(image_lp_ft)



In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))


imfilt = ax1.imshow(np.abs(image_lp), cmap='grey')
fig.colorbar(imfilt, ax=ax1)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title(r"Low pass filtered image")


imft = ax2.imshow(np.log(np.abs(np.fft.fftshift(image_lp_ft))), cmap='plasma')
fig.colorbar(imft, ax=ax2)
ax2.set_xlabel('kx')
ax2.set_ylabel('ky')
ax2.axes.xaxis.set_ticks([0,256,512])
ax2.axes.xaxis.set_ticklabels([r'$-\frac{1}{2\Delta x}$', '0', r'$\frac{1}{2\Delta x}$'])
ax2.axes.yaxis.set_ticks([0, 256, 512])
ax2.axes.yaxis.set_ticklabels([r'$-\frac{1}{2\Delta x}$', '0', r'$\frac{1}{2\Delta x}$'])
ax2.set_title(r"Low pass filtered FT")




#### 🧩Exercise, do the same with a high pass filter. What do you see?

### Smoothing  

As you can see above, the low pass filtered image present some kind of rings. This is due to the sharp cut in the filter. Using a smoother filter will remove this effect. 

A very common filter in Cosmology is the Gaussian filter, as this approximates very well the beam of the telescope. It kind of smooth out the image. 

#### 🧩Exercise, filter the image with a Gaussian kernel. What do you see?

In [ ]:
def gaussian_ft(k, sigma=5):
    """Analytic Fourier transform"""
    return np.exp(-np.pi**2 * k**2 * 2 * sigma**2)

#### 🧩 If you have time: Edge detection 

You can try to do the same with a Sobel kernel


In [ ]:
# Sobel kernel
def sobel(kx, ky):
    return 4*np.sqrt(
        (np.sin(kx)*(1+np.cos(ky)))**2 +
        (np.sin(ky)*(1+np.cos(kx)))**2
    )

